# tags

> Reading thinking and tool calls out of plain text.

Some transports cannot carry tool schemas. Models on those transports can emit `<tool_call>{json}</tool_call>` for calls and `<think>...</think>` for thinking. This module adds schemas to the system prompt and parses both batch replies and streams.

The batch and streaming parsers share the same tag tables.

In [ ]:
#| default_exp tags

In [ ]:
#| export
import json, os, re, uuid
from fastcore.all import L

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail

## Thinking

`<think>` blocks are removed before tool calls are parsed. An unterminated block means the reply ended mid-thought. Its contents remain available as thinking.

In [ ]:
#| export
_think_re = re.compile(r'<think>(.*?)</think>', re.DOTALL)

def split_think(text):
    "Split `<think>...</think>` out of `text`, returning `(clean_text, thought)`."
    text = text or ''
    ths = [m.strip() for m in _think_re.findall(text)]
    text = _think_re.sub('', text)
    if '<think>' in text:            # unterminated, e.g. cut off at the token cap
        text, _, rest = text.partition('<think>')
        ths.append(rest.strip())
    return text.strip('\n'), '\n'.join(th for th in ths if th)

In [ ]:
test_eq(split_think('<think>weighing it</think>the answer'), ('the answer', 'weighing it'))
test_eq(split_think('no tags here'), ('no tags here', ''))
test_eq(split_think('<think>one</think>mid<think>two</think>'), ('mid', 'one\ntwo'))
test_eq(split_think('answer<think>cut off here'), ('answer', 'cut off here'))
test_eq(split_think(None), ('', ''))

## A call in the text

Models use three names for the argument field. `tag_args` accepts each name. It also decodes a JSON string where an object belongs.

A model's call arrives as text. Small models often emit invalid JSON. `loose_json` accepts unescaped quotes, raw newlines, single-quoted keys, Python literals, and trailing commas. It also reports whether the object closed. An incomplete value is never returned.

In [ ]:
#| export
_ESC = {'"':'"', '\\':'\\', '/':'/', 'b':'\b', 'f':'\f', 'n':'\n', 'r':'\r', 't':'\t'}
_WS = ' \t\r\n'

def _ws(s, i):
    while i < len(s) and s[i] in _WS: i += 1
    return i

def _loose_str(s, i, ends):
    "Read a string at `s[i]`, closing only on a quote whose next non-space byte is in `ends`."
    out, i, n = [], i+1, len(s)
    while i < n:
        c = s[i]
        if c == '\\' and i+1 < n:
            e = s[i+1]
            if e in _ESC: out.append(_ESC[e]); i += 2; continue
            if e == 'u' and i+5 < n:
                try: out.append(chr(int(s[i+2:i+6], 16))); i += 6; continue
                except ValueError: pass
            out.append(e); i += 2; continue
        if c == '"':
            j = _ws(s, i+1)
            if j >= n or s[j] in ends: return ''.join(out), i+1, True
            out.append(c); i += 1; continue      # an inner quote the model never escaped
        out.append(c); i += 1                    # raw control bytes ride through as themselves
    return ''.join(out), i, False                # ran off the end: truncated

_LITS = {'true': True, 'false': False, 'null': None, 'True': True, 'False': False, 'None': None}

def _bare(tok):
    "A bare (unquoted) token as the value it spells, or as itself when it spells nothing else."
    if tok in _LITS: return _LITS[tok]
    if len(tok) > 1 and tok[0] == tok[-1] == "'": return tok[1:-1]     # a single-quoted string
    try: return int(tok)
    except ValueError: pass
    try: return float(tok)
    except ValueError: return tok

In [ ]:
#| export
def _loose_val(s, i, ends):
    "Read one value at `s[i]`, returning `(value, next_i, complete)`."
    i = _ws(s, i)
    if i >= len(s): return None, i, False
    c = s[i]
    if c == '"': return _loose_str(s, i, ends)
    if c == '{': return _loose_obj(s, i)
    if c == '[': return _loose_arr(s, i)
    j, depth = i, 0
    while j < len(s) and not (depth == 0 and s[j] in ends):    # a bare value runs to its own closer
        if s[j] in '{[': depth += 1
        elif s[j] in '}]': depth -= 1
        j += 1
    return _bare(s[i:j].strip()), j, j < len(s)

def _loose_obj(s, i):
    "Read an object at `s[i]`, returning `(dict, next_i, complete)`."
    out, i, n = {}, i+1, len(s)
    while True:
        i = _ws(s, i)
        if i >= n: return out, i, False
        if s[i] == '}': return out, i+1, True
        if s[i] == ',': i += 1; continue
        if s[i] == '"': k, i, ok = _loose_str(s, i, ':')
        else:
            j = i
            while j < n and s[j] not in ':,}': j += 1
            k, i, ok = s[i:j].strip().strip('\'"'), j, j < n
        if not ok: return out, i, False
        i = _ws(s, i)
        if i < n and s[i] == ':': i += 1
        v, i, ok = _loose_val(s, i, ',}')
        if not ok: return out, i, False        # the value was cut: it never lands in `out`
        out[k] = v

def _loose_arr(s, i):
    "Read an array at `s[i]`, returning `(list, next_i, complete)`."
    out, i, n = [], i+1, len(s)
    while True:
        i = _ws(s, i)
        if i >= n: return out, i, False
        if s[i] == ']': return out, i+1, True
        if s[i] == ',': i += 1; continue
        v, i, ok = _loose_val(s, i, ',]')
        if not ok: return out, i, False
        out.append(v)

def loose_json(s):
    "Parse `s` as a JSON object, tolerating what models get wrong. `(obj, complete)`, or `(None, False)`."
    s = (s or '').strip()
    if not s.startswith('{'): return None, False
    try: return json.loads(s), True                # strict wins, so conforming input is untouched
    except json.JSONDecodeError: pass
    obj, _, ok = _loose_obj(s, 0)
    return (obj, ok) if isinstance(obj, dict) else (None, False)

In [ ]:
test_eq(loose_json('{"a": 1}'), ({'a': 1}, True))
# an unescaped quote inside a value: the string closes at the CONFIRMED delimiter, not the first one
test_eq(loose_json('{"p": "say "hi" now"}')[0], {'p': 'say "hi" now'})
test_eq(loose_json('{"code": "a\nb"}')[0], {'code': 'a\nb'})   # a raw newline in a value
test_eq(loose_json("{'k': True, 'n': None,}")[0], {'k': True, 'n': None})
test_eq(loose_json('{"a": 1, "b": "cut')[1], False)            # ran off the end

A truncated call still names the tool it meant to call. `salvage_name` reads that name, and only from the top level: a `name` nested inside an argument is data, not the call.

In [ ]:
#| export
_name_re = re.compile(r'"name"\s*:\s*"([^"\\]{1,128})"')

def salvage_name(s):
    "The tool name from a call cut inside its own object, or None. Depth 1 only: a nested `name` is data."
    depth, i, n = 0, 0, len(s or '')
    while i < n:
        c = s[i]
        if c == '"':
            if depth == 1 and (m := _name_re.match(s, i)): return m.group(1)
            _, i, _ = _loose_str(s, i, ',}]:')
            continue
        if c in '{[': depth += 1
        elif c in '}]': depth -= 1
        i += 1
    return None

In [ ]:
#| export
def mk_tc(name, args=None):
    "A tool_call dict in the canonical OpenAI shape."
    return {'id': f'call_{uuid.uuid4().hex[:8]}', 'type': 'function',
            'function': {'name': name, 'arguments': args or {}}}

In [ ]:
test_eq(salvage_name('{"name": "edit", "arguments": {"file": "x.py", "new": "def f(:'), 'edit')
test_eq(salvage_name('{"arguments": {"name": "buried"}}'), None)   # depth 1 only
test_eq(salvage_name('no call here'), None)

In [ ]:
#| export
TAG_ARG_KEYS = ('arguments', 'input', 'parameters')

def tag_args(d):
    "The argument dict of a tagged call, decoding the JSON string some models send instead."
    for k in TAG_ARG_KEYS:
        if (v := d.get(k)) is None: continue
        if isinstance(v, str):
            v = loose_json(v)[0]
            if v is None: continue
        if isinstance(v, dict): return v
    return {}

def mk_tag_tc(s):
    "A tool_call dict from the body of a `<tool_call>` block, or None. Strict, then repaired, then salvaged."
    d, ok = loose_json(s)
    if d is not None and d.get('name'):
        return mk_tc(d['name'], tag_args(d) if ok else {})   # a cut object never ships partial values
    if (name := salvage_name(s or '')): return mk_tc(name, {})
    return None

In [ ]:
test_eq(tag_args({'arguments': {'a': 1}}), {'a': 1})
test_eq(tag_args({'input': '{"a": 1}'}), {'a': 1})     # a JSON string, decoded
test_eq(tag_args({'parameters': {'b': 2}}), {'b': 2})
test_eq(tag_args({'arguments': 'not json'}), {})
test_eq(tag_args({}), {})

In [ ]:
tc = mk_tag_tc('{"name": "add", "arguments": {"a": 1}}')
test_eq(tc['function'], {'name': 'add', 'arguments': {'a': 1}})
assert tc['id'].startswith('call_')
test_eq(mk_tag_tc('{"arguments": {}}'), None)   # no name is not a call
test_eq(mk_tag_tc('not json'), None)
test_eq(mk_tag_tc('[1, 2]'), None)

Qwen 3.5 and later use an XML tool-call dialect. XML parsing runs before JSON parsing. A `<tool_call>` body containing `<function=` is always XML. This prevents a parameter value from selecting the tool name. Only parameters with a closing `</parameter>` are returned.

In [ ]:
#| export
_fn_re   = re.compile(r'<function\s*=\s*([A-Za-z_][\w.\-]*)\s*>', re.I)
_fnend_re = re.compile(r'</function\s*>', re.I)
#: only a CLOSED parameter is a complete one, so a cut call ships the name and whatever finished
_par_re  = re.compile(r'<parameter\s*=\s*([A-Za-z_][\w.\-]*)\s*>(.*?)</parameter\s*>', re.I | re.S)
_num_re  = re.compile(r'-?\d+(?:\.\d+)?$')

def _unframe(v):
    "Drop the one framing newline the template puts on each side of a value, and nothing else."
    if v.startswith('\n'): v = v[1:]
    if v.endswith('\n'): v = v[:-1]
    return v

def _spelled(v):
    "A parameter value as the JSON it spells, else as the raw string it is."
    t = v.strip()
    if t[:1] in '{[' or t in ('true', 'false', 'null') or _num_re.match(t):
        try: return json.loads(t)
        except json.JSONDecodeError: pass
    return v

def parse_fn_tags(text):
    "Calls in the `<function=name><parameter=key>` XML dialect, which Qwen 3.5+ templates mandate."
    out = []
    for m in _fn_re.finditer(text or ''):
        body = text[m.end():]
        if (e := _fnend_re.search(body)): body = body[:e.start()]
        args = {k: _spelled(_unframe(v)) for k, v in _par_re.findall(body)}
        if not args and (d := loose_json(body)[0]): args = d
        out.append(mk_tc(m.group(1), args))
    return out

In [ ]:
tcs = parse_fn_tags('<tool_call>\n<function=read>\n<parameter=path>\na.txt\n</parameter>\n</function>\n</tool_call>')
test_eq(len(tcs), 1)
test_eq(tcs[0]['function'], {'name': 'read', 'arguments': {'path': 'a.txt'}})
# exactly one framing newline per side comes off, so a value's own indentation survives
tcs = parse_fn_tags('<function=edit>\n<parameter=old>\n    indented\n</parameter>\n</function>')
test_eq(tcs[0]['function']['arguments']['old'], '    indented')

Some models emit a bare call object without tags. The fallback accepts it only when the object is the whole reply and contains a name plus arguments. Prose that describes JSON must not execute.

In [ ]:
#| export
_res_tags = ('tool_result', 'tool_results', 'function_result', 'function_results',
             'tool_function_result', 'tool_function_results',
             'tool_function_call', 'tool_function_calls',
             'function_call', 'function_calls', 'tool_use')
_toolres_re = re.compile('|'.join(rf'</?{t}>>?' for t in _res_tags) + r'|</?tool_call>', re.I)
_toolcall_re = re.compile(r'<tool_call>\s*(.*?)\s*</tool_call>', re.DOTALL)
_fnblk_re = re.compile(r'<function\s*=.*?(?:</function\s*>|$)', re.I | re.S)
_fence_re = re.compile(r'^```(?:json)?\s*|\s*```$')

def lone_tag_tc(text, names=None):
    "A whole reply that is one bare call object: what the tags asked for, without the tags."
    s = _fence_re.sub('', (text or '').strip()).strip()
    if not (s.startswith('{') and s.endswith('}')): return None
    d, ok = loose_json(s)
    if not ok or not isinstance(d, dict) or not d.get('name'): return None
    if not any(k in d for k in TAG_ARG_KEYS): return None
    # inferred, not tagged: it must name a tool that exists, or it is the model talking about JSON
    if names is not None and d['name'] not in set(names): return None
    return mk_tc(d['name'], tag_args(d))

In [ ]:
test_eq(lone_tag_tc('{"name": "add", "arguments": {"a": 1}}')['function']['name'], 'add')
test_eq(lone_tag_tc('```json\n{"name": "add", "input": {}}\n```')['function']['name'], 'add')
test_eq(lone_tag_tc('Here is the call: {"name": "add", "arguments": {}}'), None)  # not the whole reply
test_eq(lone_tag_tc('{"name": "add"}'), None)                                    # no argument key
test_eq(lone_tag_tc('{"a": 1}'), None)
test_eq(lone_tag_tc(''), None)

`parse_tool_tags` parses tagged calls first. It tries the bare-object fallback only when no tagged calls exist. Invented result markup remains ordinary prose.

In [ ]:
#| export
def parse_tool_tags_ex(text, names=None):
    "`(clean_text, calls, failed)`. `failed` marks a call the model emitted that nothing could parse."
    t = text or ''
    tcs, failed = parse_fn_tags(t), False   # the XML dialect is read FIRST: a value never decides the call
    rest = _toolcall_re.sub('', t)
    for b in _toolcall_re.findall(t):
        if parse_fn_tags(b): continue
        if (tc := mk_tag_tc(b)): tcs.append(tc)
        else: failed = True             # a block nothing could read is a loss, never a silent deletion
    if (k := rest.rfind('<tool_call>')) >= 0:   # an opener the model never closed: cut at the token cap
        b = rest[k + len('<tool_call>'):]
        if not parse_fn_tags(b):
            if (tc := mk_tag_tc(b)): tcs.append(tc)
            else: failed = True
        rest = rest[:k]
    if not tcs and not failed and (tc := lone_tag_tc(t, names)): return '', [tc], False
    clean = _fnblk_re.sub('', _toolres_re.sub('', rest)).strip('\n')
    return clean, tcs, failed

def parse_tool_tags(text, names=None):
    "Return clean text and calls parsed from `<tool_call>` and `<function=>` blocks."
    return parse_tool_tags_ex(text, names)[:2]

In [ ]:
# four shapes that used to be dropped in silence: each must now reach the caller
mangled = 'Reading it.\n<tool_call>{"name": "read", "arguments": {"path": "a"b.txt"}}</tool_call>'
txt, tcs = parse_tool_tags(mangled)
test_eq((txt, tcs[0]['function']), ('Reading it.', {'name': 'read', 'arguments': {'path': 'a"b.txt'}}))

cut = '<tool_call>{"name": "edit", "arguments": {"file": "x.py", "new": "def f(:'
_, tcs = parse_tool_tags(cut)
test_eq(tcs[0]['function'], {'name': 'edit', 'arguments': {}})   # the name, never the half-written value

xml = '<tool_call>\n<function=read>\n<parameter=path>\na.txt\n</parameter>\n</function>\n</tool_call>'
test_eq(parse_tool_tags(xml)[1][0]['function'], {'name': 'read', 'arguments': {'path': 'a.txt'}})

strargs = '<tool_call>{"name":"read","arguments":"{path: a.txt}"}</tool_call>'
test_eq(parse_tool_tags(strargs)[1][0]['function']['arguments'], {'path': 'a.txt'})

# a block nothing can read is reported, not deleted
_, tcs, failed = parse_tool_tags_ex('<tool_call>{"totally": "unreadable"</tool_call>')
test_eq((tcs, failed), ([], True))

mixed = xml + '<tool_call>{"name": "ls", "arguments": {}}</tool_call>'
test_eq([tc['function']['name'] for tc in parse_tool_tags(mixed)[1]], ['read', 'ls'])

In [ ]:
txt, tcs = parse_tool_tags('Let me look.\n<tool_call>{"name": "ls", "arguments": {}}</tool_call>')
test_eq(txt, 'Let me look.')
test_eq([t['function']['name'] for t in tcs], ['ls'])

In [ ]:
test_eq(parse_tool_tags('just prose'), ('just prose', []))
txt, tcs = parse_tool_tags('<tool_call>{"name":"a","arguments":{}}</tool_call>'
                           '<tool_call>{"name":"b","arguments":{}}</tool_call>')
test_eq([t['function']['name'] for t in tcs], ['a', 'b'])

In [ ]:
# invented result tags go; the text stays prose, and no tool call is claimed
test_eq(parse_tool_tags('done <tool_result>42</tool_result>'), ('done 42', []))
# the bare-object fallback fires only when no tagged call was found
test_eq(parse_tool_tags('{"name": "add", "arguments": {"a": 1}}')[1][0]['function']['name'], 'add')

A model can narrate a call instead of emitting one. `tag_call_shape` detects that failure and lets the caller identify an unreliable model.

In [ ]:
#| export
def tag_call_shape(text, names=None):
    "Does `text` still show a call the parser did not take? `names` limits it to tools that exist."
    t = text or ''
    if '<tool_call' in t: return True
    if not any(f'"{k}"' in t for k in TAG_ARG_KEYS): return False
    pat = '|'.join(re.escape(str(n)) for n in (names or []) if n) if names is not None else r'[A-Za-z_]\w*'
    return bool(pat) and bool(re.search(rf'"name"\s*:\s*"(?:{pat})"', t))

In [ ]:
test_eq(tag_call_shape('I would call <tool_call> here'), True)
test_eq(tag_call_shape('maybe {"name": "add", "arguments": {}}', ['add']), True)
test_eq(tag_call_shape('maybe {"name": "add", "arguments": {}}', ['other']), False)  # no such tool
test_eq(tag_call_shape('plain prose'), False)
test_eq(tag_call_shape('{"name": "add"}'), False)   # no argument key, so not call-shaped

## Putting schemas in the prompt

`tag_tool_prompt` describes tools to a model when the transport cannot send schemas.

In [ ]:
#| export
TAG_TOOLS_SP = """

# Tools

You can call the functions below. Their signatures are given as JSON schemas inside \
<tools></tools>:

<tools>
{tools}
</tools>

To call one, emit a JSON object with the function's name and its arguments inside \
<tool_call></tool_call>. Then end your reply immediately and wait:

<tool_call>
{{"name": "the_function_name", "arguments": {{"first": "value"}}}}
</tool_call>

Nothing may follow </tool_call> in the same message: not a comment, not a guess at what the \
result will be, not another call. Stop there. Call one function at a time.

Do not describe the call in prose as well as emitting it, and never invent a result -- the real \
one comes back in the next message, under a "## Tool result (name)" heading. That heading is the \
only form a result ever takes: do not write result markup of your own, do not wrap a result in \
tags, and do not copy a result back into your reply. Say what you concluded from it, not what \
it said."""

def tag_tools_sp(toolspecs, sp='', template=TAG_TOOLS_SP):
    "`sp` plus the tag protocol and `toolspecs` as JSON, for a transport that can't carry tools."
    if not toolspecs: return sp
    block = '\n'.join(json.dumps(t, ensure_ascii=False) for t in toolspecs)
    return (sp or '') + template.format(tools=block)

In [ ]:
spec = {'type': 'function', 'function': {'name': 'add', 'parameters': {}}}
out = tag_tools_sp([spec], 'Be brief.')
assert out.startswith('Be brief.') and '"name": "add"' in out and '<tool_call>' in out
test_eq(tag_tools_sp([], 'Be brief.'), 'Be brief.')   # no tools, no protocol

## Streaming

A stream can split a tag between deltas. `StreamSplit` retains any buffer suffix that could become a tag and resumes when the next delta arrives. It withholds raw `<tool_call>` JSON and emits the parsed call instead.

Hand-written cases do not cover every model output. Set `$URAI_RAW_DUMP` to append replies with tool calls and their schemas as JSON lines. `tests/test_tool_traffic.py` replays the file through parsing and coercion. When disabled, capture costs one environment read per turn.

In [ ]:
#| export
def RAW_DUMP():
    "Path a capture is writing to, from `$URAI_RAW_DUMP`, or None. Read per call, so a test can set it."
    return os.environ.get('URAI_RAW_DUMP') or None

def dump_raw(toolspecs, raw, path=None):
    "Append one `(tools, raw)` pair to the capture file, for replaying real traffic through the parser."
    if not (path := path or RAW_DUMP()) or not raw: return False
    with open(path, 'a') as f: f.write(json.dumps({'tools': toolspecs or [], 'raw': raw}) + '\n')
    return True


In [ ]:
#| export
_tags = ('<think>', '</think>', '<tool_call>', '</tool_call>', '<function=', '</function>',
         *(f'<{n}>' for n in _res_tags), *(f'</{n}>' for n in _res_tags))

class StreamSplit:
    "Stateful splitter: `<think>` becomes thought chunks, tool blocks are held back and parsed."
    def __init__(self, keep_raw=None):
        self.buf, self.state, self.text, self.thought = '', 'text', '', ''
        self.tool_calls, self._tc_buf, self._strip = [], '', False
        self.failed, self._end = False, '</tool_call>'
        self.raw = '' if (RAW_DUMP() if keep_raw is None else keep_raw) else None

    def _held(self):
        "Length of the longest `buf` suffix that could still become a tag."
        for n in range(min(len(self.buf), max(map(len, _tags)) - 1), 0, -1):
            if any(t.startswith(self.buf[-n:]) for t in _tags): return n
        return 0

    def _emit_text(self, out):
        out = _toolres_re.sub('', out)       # invented result markup: the streamed path never
        if self._strip: out = out.lstrip('\n')   # reaches `parse_tool_tags`, so it is dropped here
        if not out: return None
        self._strip = False; self.text += out
        return {'content': [{'type': 'text', 'text': out}]}

    def _close_tool(self):
        "Run the buffered tool block through the same ladder the non-streaming path uses."
        buf, self._tc_buf = self._tc_buf, ''
        if (tcs := parse_fn_tags(buf)): self.tool_calls += tcs
        elif (tc := mk_tag_tc(buf)): self.tool_calls.append(tc)
        elif buf.strip(): self.failed = True

    def feed(self, s):
        "Consume a text delta and yield chunk dicts."
        if self.raw is not None: self.raw += s
        self.buf += s
        while True:
            if self.state == 'text':
                cands = [(k, t) for k, t in ((self.buf.find('<think>'), '<think>'),
                                             (self.buf.find('<tool_call>'), '<tool_call>'),
                                             (self.buf.find('<function='), '<function=')) if k >= 0]
                if not cands:
                    n = self._held()
                    out, self.buf = self.buf[:len(self.buf) - n], self.buf[len(self.buf) - n:]
                    if (c := self._emit_text(out)): yield c
                    return
                k, tag = min(cands)
                out, self.buf = self.buf[:k], self.buf[k + len(tag):]
                if tag == '<think>': self.state = 'think'
                else:
                    self.state, self._end = 'tool', '</tool_call>' if tag == '<tool_call>' else '</function>'
                    # the opener is part of the block the XML parser reads, so it goes in the buffer
                    self._tc_buf = '' if tag == '<tool_call>' else tag
                if (c := self._emit_text(out)): yield c
            elif self.state == 'think':
                k = self.buf.find('</think>')
                if k < 0:
                    n = self._held()
                    out, self.buf = self.buf[:len(self.buf) - n], self.buf[len(self.buf) - n:]
                    if out: self.thought += out; yield {'channels': {'thought': out}}
                    return
                out, self.buf, self.state, self._strip = self.buf[:k], self.buf[k + len('</think>'):], 'text', True
                if out: self.thought += out; yield {'channels': {'thought': out}}
            else:                                    # inside a tool call
                k = self.buf.find(self._end)
                if k < 0:
                    n = self._held()
                    self._tc_buf += self.buf[:len(self.buf) - n]; self.buf = self.buf[len(self.buf) - n:]
                    return
                self._tc_buf += self.buf[:k] + (self._end if self._end == '</function>' else '')
                self.buf, self.state, self._strip = self.buf[k + len(self._end):], 'text', True
                self._close_tool()

    def finish(self):
        "Flush leftovers: unterminated think becomes thought, an unclosed tool block is salvaged."
        s, self.buf = self.buf, ''
        if self.state == 'think':
            if s: self.thought += s; yield {'channels': {'thought': s}}
        elif self.state == 'tool':
            self._tc_buf += s
            self._close_tool()
        elif (c := self._emit_text(s)): yield c

In [ ]:
def _run(deltas):
    "Feed `deltas` through a fresh splitter and return it with the chunks it produced."
    s = StreamSplit()
    chunks = [c for d in deltas for c in s.feed(d)] + list(s.finish())
    return s, chunks

s, _ = _run(['<think>', 'wei', 'ghing it', '</think>', 'the ', 'answer'])
test_eq((s.thought, s.text), ('weighing it', 'the answer'))

In [ ]:
# a tag split across three deltas is still one tag
s, _ = _run(['<thi', 'nk>deep</th', 'ink>done'])
test_eq((s.thought, s.text), ('deep', 'done'))

In [ ]:
s, _ = _run(['Looking.<tool_call>{"name": "ls", ', '"arguments": {}}</tool_call>'])
test_eq(s.text, 'Looking.')                       # the JSON never reaches the caller as text
test_eq([t['function']['name'] for t in s.tool_calls], ['ls'])

In [ ]:
s, _ = _run(['<think>cut off mid-thought'])       # unterminated: flushed by `finish`
test_eq((s.thought, s.text), ('cut off mid-thought', ''))
s, _ = _run(['<tool_call>{"name": "ls", "arguments": {}}'])   # unclosed, but parseable
test_eq([t['function']['name'] for t in s.tool_calls], ['ls'])

In [ ]:
_, chunks = _run(['<think>why</think>because'])
test_eq(chunks, [{'channels': {'thought': 'why'}},
                 {'content': [{'type': 'text', 'text': 'because'}]}])
test_eq(_run(['plain text'])[0].text, 'plain text')

Native tool calls can also span deltas. OpenAI identifies their fragments by index. `acc_tc` combines the fragments into complete calls.

In [ ]:
#| export
def acc_tc(acc, deltas):
    "Fold streamed OpenAI `tool_calls` deltas into `acc`, a list of partial tool_call dicts."
    for d in deltas or []:
        i = d.get('index', 0)
        while len(acc) <= i: acc.append({'id': None, 'type': 'function',
                                         'function': {'name': '', 'arguments': ''}})
        if d.get('id'): acc[i]['id'] = d['id']
        f = d.get('function') or {}
        if f.get('name'): acc[i]['function']['name'] += f['name']
        if f.get('arguments'): acc[i]['function']['arguments'] += f['arguments']

In [ ]:
acc = []
acc_tc(acc, [{'index': 0, 'id': 'c1', 'function': {'name': 'ad'}}])
acc_tc(acc, [{'index': 0, 'function': {'name': 'd', 'arguments': '{"a":'}}])
acc_tc(acc, [{'index': 0, 'function': {'arguments': ' 1}'}}])
test_eq(acc, [{'id': 'c1', 'type': 'function', 'function': {'name': 'add', 'arguments': '{"a": 1}'}}])

In [ ]:
acc = []
acc_tc(acc, [{'index': 1, 'id': 'c2', 'function': {'name': 'b'}}])   # index 1 arrives first
test_eq(len(acc), 2)
test_eq(acc[0]['function']['name'], '')       # index 0 is a placeholder until it arrives
test_eq(acc[1]['function']['name'], 'b')
acc_tc(acc, None)
test_eq(len(acc), 2)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()